# 01 — Dipole field, force, torque, and source-formula check

**Question.** Does the closed-form force in the supplied appendix agree with the gradient of magnetic potential energy?

We use a convention that removes ambiguity:

- `r = p_receiver - p_source` points from the EPM to the IPM;
- `m_source` is the EPM moment and `m_receiver` is the IPM moment;
- returned force and torque act on the IPM.

The independent identities are

$$
\mathbf B=\frac{\mu_0}{4\pi r^3}(3\hat{\mathbf r}\hat{\mathbf r}^T-I)\mathbf m_E,
\quad U=-\mathbf m_I^T\mathbf B,
$$

$$
\mathbf F=-\nabla_{\mathbf r}U,
\qquad \boldsymbol\tau=\mathbf m_I\times\mathbf B.
$$


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.magnetics import (
    MU0, dipole_field, dipole_force, magnetic_energy,
    magnetic_torque, normalize, numerical_gradient,
)
np.set_printoptions(precision=8, suppress=False)


## A generic nonsymmetric test pose

Axis-aligned examples can accidentally make an incorrect cross term vanish. We therefore use nonparallel magnet moments and an oblique relative position.


In [ ]:
r = np.array([0.071, -0.026, 0.113])            # m, EPM -> IPM
m_E = 2.40 * normalize(np.array([0.34, 0.18, 0.92]))  # A m^2
m_I = 0.055 * normalize(np.array([-0.25, 0.83, 0.49]))

B = dipole_field(r, m_E)
F_closed = dipole_force(r, m_E, m_I)
F_energy = -numerical_gradient(
    lambda x: magnetic_energy(x, m_E, m_I), r, step=2e-7
)
T = magnetic_torque(r, m_E, m_I)

print("B [T]       =", B)
print("F closed [N]=", F_closed)
print("F=-grad U [N]=", F_energy)
print("Torque [Nm] =", T)
print("relative force error =", np.linalg.norm(F_closed-F_energy)/np.linalg.norm(F_energy))
assert np.allclose(F_closed, F_energy, rtol=2e-8)


## Matrix form and the apparent printed cross-term error

Let $\mathbf m_E=m_E\hat{\mathbf e}_E$, $\mathbf m_I=m_I\hat{\mathbf e}_I$,
$Z=I-5\hat{\mathbf r}\hat{\mathbf r}^T$, and
$\lambda=\mu_0m_Em_I/(4\pi r^3)$. The energy-consistent matrix expression is

$$
\mathbf F=\frac{3\lambda}{r}
[\hat e_I\hat e_E^T+\hat e_E\hat e_I^T+(\hat e_I^TZ\hat e_E)I]\hat r.
$$

The printed appendix appears to repeat $\hat e_I$ in the scalar term. We compute both versions.


In [ ]:
distance = np.linalg.norm(r)
r_hat = r / distance
e_E, e_I = normalize(m_E), normalize(m_I)
lambda_ = MU0 * np.linalg.norm(m_E) * np.linalg.norm(m_I) / (4*np.pi*distance**3)
Z = np.eye(3) - 5*np.outer(r_hat, r_hat)

common = np.outer(e_I, e_E) + np.outer(e_E, e_I)
F_cross = 3*lambda_/distance * (
    common + (e_I @ Z @ e_E)*np.eye(3)
) @ r_hat
F_printed = 3*lambda_/distance * (
    common + (e_I @ Z @ e_I)*np.eye(3)
) @ r_hat

for label, force in [("energy gradient", F_energy),
                     ("standard cross term", F_cross),
                     ("printed repeated-IPM term", F_printed)]:
    print(f"{label:27s}", force)
print("standard relative error:", np.linalg.norm(F_cross-F_energy)/np.linalg.norm(F_energy))
print("printed relative error :", np.linalg.norm(F_printed-F_energy)/np.linalg.norm(F_energy))
assert np.allclose(F_cross, F_energy, rtol=2e-8)
assert not np.allclose(F_printed, F_energy, rtol=1e-3)


## Surgical-depth scaling

For fixed moments and orientation, field and torque scale as $r^{-3}$ while force scales as $r^{-4}$. The plot normalizes both to their values at 50 mm. It visualizes why abdominal-wall thickness and target depth dominate permanent-magnet design.


In [ ]:
distances_mm = np.linspace(50, 250, 401)
torque_ratio = (50 / distances_mm)**3
force_ratio = (50 / distances_mm)**4

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.semilogy(distances_mm, torque_ratio, label=r"field / torque $r^{-3}$", lw=2.5)
ax.semilogy(distances_mm, force_ratio, label=r"force $r^{-4}$", lw=2.5)
ax.axvspan(100, 180, color="tab:orange", alpha=.12, label="example human-scale depth band")
ax.set(xlabel="EPM–IPM separation [mm]", ylabel="ratio relative to 50 mm",
       title="Distance penalty for dipole actuation")
ax.grid(True, which="both", alpha=.3)
ax.legend()
plt.show()


## Interpretation

- The energy gradient is the independent reference, not another rearrangement of the same force equation.
- The standard cross term matches that reference at a generic pose.
- The expression with two IPM axes generally does not.
- A model that is adequate at 50 mm can become physically weak at 150–200 mm even before robot reach, collisions, or tissue are considered.
